In this notebook we will take a closer look into the Random Forest algorithm. First lets do a Random Forest without any hyperparameter tuning and then we will do a Random Forest with hyperparameter tuning.


In [12]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import SelectKBest, f_classif

In [13]:

# Load the dataset
df = pd.read_csv('winequality-red-clean.csv')
X = df.drop('quality', axis=1)  # Features
y = df['quality']  # Target (quality)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [14]:
# Feature Selection: Select Top 5 Features Using SelectKBest
k = 5  # Number of top features to select
selector = SelectKBest(score_func=f_classif, k=k)

# Fit to training data
X_train = selector.fit_transform(X_train, y_train)
X_test = selector.transform(X_test)  # Apply same transformation to test data

# Print selected feature names
selected_features = selector.get_support(indices=True)
print(f"Selected top {k} features: {X.columns[selected_features]}")


Selected top 5 features: Index(['volatile acidity', 'citric acid', 'total sulfur dioxide', 'sulphates',
       'alcohol'],
      dtype='object')


In [15]:

# Initialize a DataFrame to store performance results
performance_df = pd.DataFrame(columns=[
    "Model",
    "Train Accuracy",
    "Test Accuracy",
    "Cross-Validation Accuracy (Mean ± Std)",
    "Bias",
    "Variance"
])

# 1. Random Forest without Hyperparameter Tuning
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)

# Predict on training and test data
y_train_pred = rf_model.predict(X_train)
y_test_pred = rf_model.predict(X_test)

# Calculate accuracy for both train and test sets
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

# Cross-validation results (5-fold)
cv_scores = cross_val_score(rf_model, X_train, y_train, cv=5)
cv_mean = cv_scores.mean()
cv_std = cv_scores.std()

# Calculate Bias and Variance
bias = 100 - (train_accuracy * 100)  # Bias as 100 - train accuracy
variance = (train_accuracy - test_accuracy) * 100  # Variance as the difference between train and test accuracy

# Store the results in the performance_df DataFrame
performance_df.loc[len(performance_df.index)] = [
    "Random Forest (No Tuning)",
    f"{train_accuracy * 100:.2f}%",
    f"{test_accuracy * 100:.2f}%",
    f"{cv_mean * 100:.2f}% ± {cv_std * 100:.2f}%",
    f"{bias:.2f}%",
    f"{variance:.2f}%"
]

performance_df


,Model,Train Accuracy,Test Accuracy,Cross-Validation Accuracy (Mean ± Std),Bias,Variance
0,Random Forest (No Tuning),100.00%,69.52%,65.95% ± 2.36%,0.00%,30.48%


In [16]:

# 2. Random Forest with Hyperparameter Tuning
# 2. Random Forest with Hyperparameter Tuning
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'max_features': ['sqrt', 'log2'],  # Added max_features parameter
}

# Initialize GridSearchCV
grid_search = GridSearchCV(estimator=RandomForestClassifier(random_state=42),
                           param_grid=param_grid,
                           scoring='accuracy',
                           cv=5,
                           n_jobs=-1)

# Fit the model
grid_search.fit(X_train, y_train)
best_rf_model = grid_search.best_estimator_

# Predict on training and test data
y_train_pred_tuned = best_rf_model.predict(X_train)
y_test_pred_tuned = best_rf_model.predict(X_test)

# Calculate accuracy for both train and test sets
train_accuracy_tuned = accuracy_score(y_train, y_train_pred_tuned)
test_accuracy_tuned = accuracy_score(y_test, y_test_pred_tuned)

# Cross-validation results (5-fold)
cv_scores_tuned = cross_val_score(best_rf_model, X_train, y_train, cv=5)
cv_mean_tuned = cv_scores_tuned.mean()
cv_std_tuned = cv_scores_tuned.std()

# Calculate Bias and Variance
bias_tuned = 100 - (train_accuracy_tuned * 100)  # Bias as 100 - train accuracy
variance_tuned = (
                             train_accuracy_tuned - test_accuracy_tuned) * 100  # Variance as the difference between train and test accuracy

# Store the results in the performance_df DataFrame
performance_df.loc[len(performance_df.index)] = [
    "Random Forest (With Tuning)",
    f"{train_accuracy_tuned * 100:.2f}%",
    f"{test_accuracy_tuned * 100:.2f}%",
    f"{cv_mean_tuned * 100:.2f}% ± {cv_std_tuned * 100:.2f}%",
    f"{bias_tuned:.2f}%",
    f"{variance_tuned:.2f}%"
]

# Display the results table
best_params = grid_search.best_params_
print(f"Best Params : {best_params}")
performance_df


Best Params : {'max_depth': 20, 'max_features': 'sqrt', 'min_samples_split': 5, 'n_estimators': 200}


,Model,Train Accuracy,Test Accuracy,Cross-Validation Accuracy (Mean ± Std),Bias,Variance
0,Random Forest (No Tuning),100.00%,69.52%,65.95% ± 2.36%,0.00%,30.48%
1,Random Forest (With Tuning),97.77%,68.84%,66.64% ± 2.78%,2.23%,28.93%


Here we can see that the Random Forest model with hyperparameter tuning has a little bit better test accuracy compared to the Random Forest model without hyperparameter tuning. The bias and variance are also lower for the model with hyperparameter tuning. Both are overfitting the data, but the model with hyperparameter tuning is overfitting less. Let's see what we can do about that.

In [17]:
# 3. Random Forest with Hyperparameter Tuning and Regularization
#  Set up the parameter grid for tuning with regularization
param_grid = {
    'n_estimators': [50, 100],  # Number of trees in the forest
    'max_depth': [None, 10, 20],  # Maximum depth of the tree
    'min_samples_split': [2, 5],  # Minimum number of samples required to split an internal node
    'min_samples_leaf': [1, 2],  # Minimum number of samples required to be at a leaf node
    'max_features': ['log2', 'sqrt']  # Number of features to consider for the best split
}

# Initialize GridSearchCV
grid_search = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5)

# Fit the model with hyperparameter tuning
grid_search.fit(X_train, y_train)

# Get the best model from grid search
best_rf_model = grid_search.best_estimator_

# Predict on training data and test data with the best model
y_train_pred_tuning = best_rf_model.predict(X_train)
y_test_pred_tuning = best_rf_model.predict(X_test)

# Accuracy for both train and test sets
train_accuracy_tuning = accuracy_score(y_train, y_train_pred_tuning)
test_accuracy_tuning = accuracy_score(y_test, y_test_pred_tuning)

# Cross-validation results (5-fold) with the best model
cv_scores_tuning = cross_val_score(best_rf_model, X_train, y_train, cv=5)
cv_mean_tuning = cv_scores_tuning.mean()
cv_std_tuning = cv_scores_tuning.std()

# Calculate Bias and Variance
bias_tuning = 100 - (train_accuracy_tuning * 100)  # Bias as 100 - train accuracy
variance_tuning = (train_accuracy_tuning - test_accuracy_tuning) * 100  # Variance

# Store the results in the performance_df DataFrame
performance_df.loc[len(performance_df.index)] = [
    "Random Forest (With Tuning and Regularization)",
    f"{train_accuracy_tuning * 100:.2f}%",
    f"{test_accuracy_tuning * 100:.2f}%",
    f"{cv_mean_tuning * 100:.2f}% ± {cv_std_tuning * 100:.2f}%",
    f"{bias_tuning:.2f}%",
    f"{variance_tuning:.2f}%"
]

# Display the performance results
best_params = grid_search.best_params_
print(f"Best Params : {best_params}")
performance_df

Best Params : {'max_depth': None, 'max_features': 'log2', 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 100}


,Model,Train Accuracy,Test Accuracy,Cross-Validation Accuracy (Mean ± Std),Bias,Variance
0,Random Forest (No Tuning),100.00%,69.52%,65.95% ± 2.36%,0.00%,30.48%
1,Random Forest (With Tuning),97.77%,68.84%,66.64% ± 2.78%,2.23%,28.93%
2,Random Forest (With Tuning and Regularization),94.77%,69.18%,66.38% ± 3.35%,5.23%,25.59%


We managed to get the variance down a bit, although it is still quite high. The bias is also lower than before. We can try to reduce the variance even more by reducing the number of features. Let's try that.

In [18]:
# from sklearn.feature_selection import SelectKBest, f_classif
# from imblearn.pipeline import Pipeline
# 
# # 4. Random Forest with Hyperparameter Tuning, Regularization, and Feature Reduction
# 
# pipeline = Pipeline([
#     ('feature_selection', SelectKBest(score_func=f_classif)),
#     ('classifier', RandomForestClassifier(random_state=42, class_weight='balanced'))
# ])
# 
# # Set the hyperparameters for tuning: testing feature counts from 1 to 10
# param_grid = {
#     'feature_selection__k': list(range(1, 11)),  # Testing all numbers from 1 to 10
#     'classifier__n_estimators': [50, 100],  # Number of trees in the forest
#     'classifier__max_depth': [None, 10, 20],  # Max depth of the trees
#     'classifier__min_samples_split': [2, 5],  # Minimum samples required to split an internal node
#     'classifier__min_samples_leaf': [1, 2],  # Minimum samples required to be at a leaf node
# }
# 
# # Perform Grid Search with cross-validation
# grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=1, refit='accuracy')
# grid_search.fit(X_train, y_train)
# 
# # Get the best estimator
# best_model_tuning = grid_search.best_estimator_
# 
# # Predict on training and test sets
# y_train_pred_tuning = best_model_tuning.predict(X_train)
# y_test_pred_tuning = best_model_tuning.predict(X_test)
# 
# # Accuracy for training and testing sets
# train_accuracy_tuning = accuracy_score(y_train, y_train_pred_tuning)
# test_accuracy_tuning = accuracy_score(y_test, y_test_pred_tuning)
# 
# # Cross-validation results
# cv_scores_tuning = cross_val_score(best_model_tuning, X_train, y_train, cv=5)
# cv_mean_tuning = cv_scores_tuning.mean()
# cv_std_tuning = cv_scores_tuning.std()
# 
# # Calculate Bias and Variance
# bias_tuning = 100 - (train_accuracy_tuning * 100)
# variance_tuning = (train_accuracy_tuning - test_accuracy_tuning) * 100
# 
# # Store the results
# performance_df.loc[len(performance_df.index)] = [
#     "Random Forest (Feature Reduction)",
#     f"{train_accuracy_tuning * 100:.2f}%",
#     f"{test_accuracy_tuning * 100:.2f}%",
#     f"{cv_mean_tuning * 100:.2f}% ± {cv_std_tuning * 100:.2f}%",
#     f"{bias_tuning:.2f}%",
#     f"{variance_tuning:.2f}%"
# ]
# 
# # Display the performance DataFrame
# performance_df

That didn't help much. We can try using a technique called "Out-Of-Bag (OOB) Error" to reduce the variance. Let's try that.

### How does OOB Error work?
Bootstrap Sampling:
 When training a Random Forest, multiple decision trees are created using different bootstrap samples from the original dataset. A bootstrap sample is generated by randomly sampling with replacement from the dataset.

OOB Samples: Since the sample is drawn with replacement, about one-third of the data points are left out of each bootstrap sample. These points are the OOB samples for that particular tree.

Prediction Using OOB Samples: When making predictions, the OOB samples can be fed into the trees that did not use them during training. This allows you to assess the performance of the model on data it hasn’t seen.

Error Estimation: By aggregating the predictions from all the trees that did not use a particular observation, you can compute an OOB prediction for that observation. The OOB error is then calculated based on the discrepancies between the predicted and actual values across all observations.

In [19]:
# # 5. Random Forest with Early Stopping
# param_grid = {
#     'n_estimators': [50, 100, 200],
#     'max_depth': [None, 10, 20, 30],
#     'min_samples_split': [2, 5, 10],
#     'min_samples_leaf': [1, 2, 4],
#     'max_features': ['sqrt', 'log2'],
#     'oob_score': [True]  # Enable OOB scoring
# }
# 
# # Initialize the model
# rf_model = RandomForestClassifier(random_state=42)
# 
# # Initialize GridSearchCV for hyperparameter tuning
# grid_search = GridSearchCV(rf_model, param_grid, cv=5, scoring='accuracy')
# 
# # Fit the model to the training data
# grid_search.fit(X_train, y_train)
# 
# # Best model parameters
# best_params = grid_search.best_params_
# 
# # Evaluate the model on the validation set using OOB score
# val_accuracy = grid_search.best_estimator_.oob_score_
# 
# # Train final model on the full training set with the best parameters
# final_model = RandomForestClassifier(**best_params, random_state=42)
# final_model.fit(X_train, y_train)
# 
# # Evaluate on the test set
# y_test_pred = final_model.predict(X_test)
# test_accuracy = accuracy_score(y_test, y_test_pred)
# 
# # Cross-validation results
# cv_scores = grid_search.cv_results_['mean_test_score']
# cv_std = grid_search.cv_results_['std_test_score']
# cv_mean = cv_scores.mean()
# cv_std_mean = cv_std.mean()
# 
# # Calculate Bias and Variance
# bias = 100 - (val_accuracy * 100)  # Bias as 100 - OOB validation accuracy
# variance = (val_accuracy - test_accuracy) * 100  # Variance as the difference between OOB and test accuracy
# 
# 
# performance_df.loc[len(performance_df.index)] = [
#     "Random Forest (With Tuning and OOB)",
#     f"{(grid_search.best_score_ * 100):.2f}%",
#     f"{test_accuracy * 100:.2f}%",
#     f"{cv_mean * 100:.2f}% ± {cv_std_mean * 100:.2f}%",
#     f"{bias:.2f}%",
#     f"{variance:.2f}%"
# ]
# performance_df

Yay!! We managed to reduce the variance quite a bit and our model is not overfitting like crazy anymore and the testing accuracy didn't drop much. 

In [20]:
# Import AdaBoost Classifier
from sklearn.ensemble import AdaBoostClassifier

# Initialize a Random Forest model as the base estimator
base_rf_model = RandomForestClassifier(n_estimators=50, random_state=42)

# Initialize the AdaBoostClassifier with the RandomForestClassifier as the base estimator
adaboost_rf_model = AdaBoostClassifier(
    estimator=base_rf_model,
    n_estimators=50,  # Number of boosting rounds
    learning_rate=1.0,  # Weight applied to each classifier
    random_state=42
)

# Fit the AdaBoost model with the RandomForest base estimator
adaboost_rf_model.fit(X_train, y_train)

# Predict on training and test data
y_train_pred_ada = adaboost_rf_model.predict(X_train)
y_test_pred_ada = adaboost_rf_model.predict(X_test)

# Calculate accuracy for both train and test sets
train_accuracy_ada = accuracy_score(y_train, y_train_pred_ada)
test_accuracy_ada = accuracy_score(y_test, y_test_pred_ada)

# Cross-validation results (5-fold)
cv_scores_ada = cross_val_score(adaboost_rf_model, X_train, y_train, cv=5)
cv_mean_ada = cv_scores_ada.mean()
cv_std_ada = cv_scores_ada.std()

# Calculate Bias and Variance
bias_ada = 100 - (train_accuracy_ada * 100)  # Bias as 100 - train accuracy
variance_ada = (
                           train_accuracy_ada - test_accuracy_ada) * 100  # Variance as the difference between train and test accuracy

# Store the results in the performance_df DataFrame
performance_df.loc[len(performance_df.index)] = [
    "Random Forest with AdaBoost",
    f"{train_accuracy_ada * 100:.2f}%",
    f"{test_accuracy_ada * 100:.2f}%",
    f"{cv_mean_ada * 100:.2f}% ± {cv_std_ada * 100:.2f}%",
    f"{bias_ada:.2f}%",
    f"{variance_ada:.2f}%"
]

# Display the performance results
performance_df


/home/omarhammad/IdeaProjects/data_5/venv/lib/python3.12/site-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/home/omarhammad/IdeaProjects/data_5/venv/lib/python3.12/site-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/home/omarhammad/IdeaProjects/data_5/venv/lib/python3.12/site-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/home/omarhammad/IdeaProjects/data_5/venv/lib/python3.12/site-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecat

,Model,Train Accuracy,Test Accuracy,Cross-Validation Accuracy (Mean ± Std),Bias,Variance
0,Random Forest (No Tuning),100.00%,69.52%,65.95% ± 2.36%,0.00%,30.48%
1,Random Forest (With Tuning),97.77%,68.84%,66.64% ± 2.78%,2.23%,28.93%
2,Random Forest (With Tuning and Regularization),94.77%,69.18%,66.38% ± 3.35%,5.23%,25.59%
3,Random Forest with AdaBoost,100.00%,67.81%,66.64% ± 3.06%,0.00%,32.19%


In [21]:
from sklearn.ensemble import ExtraTreesClassifier

# 1. Extra Trees without Hyperparameter Tuning
extra_trees_model = ExtraTreesClassifier(random_state=42)
extra_trees_model.fit(X_train, y_train)

# Predict on training and test data
y_train_pred = extra_trees_model.predict(X_train)
y_test_pred = extra_trees_model.predict(X_test)

# Calculate accuracy for both train and test sets
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

# Cross-validation results (5-fold)
cv_scores = cross_val_score(extra_trees_model, X_train, y_train, cv=5)
cv_mean = cv_scores.mean()
cv_std = cv_scores.std()

# Calculate Bias and Variance
bias = 100 - (train_accuracy * 100)  # Bias as 100 - train accuracy
variance = (train_accuracy - test_accuracy) * 100  # Variance

# Store the results in the performance_df DataFrame
performance_df.loc[len(performance_df.index)] = [
    "Extra Trees (No Tuning)",
    f"{train_accuracy * 100:.2f}%",
    f"{test_accuracy * 100:.2f}%",
    f"{cv_mean * 100:.2f}% ± {cv_std * 100:.2f}%",
    f"{bias:.2f}%",
    f"{variance:.2f}%"
]

# Display the performance results
performance_df

,Model,Train Accuracy,Test Accuracy,Cross-Validation Accuracy (Mean ± Std),Bias,Variance
0,Random Forest (No Tuning),100.00%,69.52%,65.95% ± 2.36%,0.00%,30.48%
1,Random Forest (With Tuning),97.77%,68.84%,66.64% ± 2.78%,2.23%,28.93%
2,Random Forest (With Tuning and Regularization),94.77%,69.18%,66.38% ± 3.35%,5.23%,25.59%
3,Random Forest with AdaBoost,100.00%,67.81%,66.64% ± 3.06%,0.00%,32.19%
4,Extra Trees (No Tuning),100.00%,68.84%,65.61% ± 2.81%,0.00%,31.16%


In [22]:
# Hyperparameter tuning for Extra Trees Classifier
param_grid = {
    'n_estimators': [50, 100, 200],  # Number of trees in the forest
    'max_depth': [None, 10, 20, 30],  # Maximum depth of the trees
    'min_samples_split': [2, 5, 10],  # Minimum number of samples required to split an internal node
    'min_samples_leaf': [1, 2, 4],  # Minimum number of samples required to be at a leaf node
    'max_features': ['sqrt', 'log2'],  # Number of features to consider when looking for the best split
}

# Initialize GridSearchCV
grid_search = GridSearchCV(ExtraTreesClassifier(random_state=42), param_grid, cv=5, n_jobs=-1, scoring='accuracy')

# Fit the model with hyperparameter tuning
grid_search.fit(X_train, y_train)

# Get the best model from grid search
best_extra_trees_model = grid_search.best_estimator_

# Predict on training and test data with the best model
y_train_pred_tuned = best_extra_trees_model.predict(X_train)
y_test_pred_tuned = best_extra_trees_model.predict(X_test)

# Calculate accuracy for both train and test sets
train_accuracy_tuned = accuracy_score(y_train, y_train_pred_tuned)
test_accuracy_tuned = accuracy_score(y_test, y_test_pred_tuned)

# Cross-validation results (5-fold) with the best model
cv_scores_tuned = cross_val_score(best_extra_trees_model, X_train, y_train, cv=5)
cv_mean_tuned = cv_scores_tuned.mean()
cv_std_tuned = cv_scores_tuned.std()

# Calculate Bias and Variance
bias_tuned = 100 - (train_accuracy_tuned * 100)  # Bias as 100 - train accuracy
variance_tuned = (train_accuracy_tuned - test_accuracy_tuned) * 100  # Variance

# Store the results in the performance_df DataFrame
performance_df.loc[len(performance_df.index)] = [
    "Extra Trees (With Tuning)",
    f"{train_accuracy_tuned * 100:.2f}%",
    f"{test_accuracy_tuned * 100:.2f}%",
    f"{cv_mean_tuned * 100:.2f}% ± {cv_std_tuned * 100:.2f}%",
    f"{bias_tuned:.2f}%",
    f"{variance_tuned:.2f}%"
]

# Display the performance results
performance_df

/home/omarhammad/IdeaProjects/data_5/venv/lib/python3.12/site-packages/numpy/ma/core.py:2820: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


,Model,Train Accuracy,Test Accuracy,Cross-Validation Accuracy (Mean ± Std),Bias,Variance
0,Random Forest (No Tuning),100.00%,69.52%,65.95% ± 2.36%,0.00%,30.48%
1,Random Forest (With Tuning),97.77%,68.84%,66.64% ± 2.78%,2.23%,28.93%
2,Random Forest (With Tuning and Regularization),94.77%,69.18%,66.38% ± 3.35%,5.23%,25.59%
3,Random Forest with AdaBoost,100.00%,67.81%,66.64% ± 3.06%,0.00%,32.19%
4,Extra Trees (No Tuning),100.00%,68.84%,65.61% ± 2.81%,0.00%,31.16%
5,Extra Trees (With Tuning),96.23%,68.84%,67.75% ± 1.44%,3.77%,27.39%
